In [1]:
import pandas as pd
# import ast
# import re

In [2]:
%reload_ext autoreload
%autoreload 2

import sys
sys.path.append("../../libs")

from utils import ios
from utils import constants as cons
from utils import text as txtlib

In [3]:
RESULTS_PATH = '../../../results/responses/results_<source>_<language>'

In [4]:
df_results = pd.DataFrame()
df_summary = pd.DataFrame()
df_recommendations = pd.DataFrame()

for source in cons.LLM_SOURCES:

    for language in cons.LANGUAGES:
        
        path = RESULTS_PATH.replace('<source>', source).replace('<language>', language)

        if ios.path_exists(path):
            prefix = f"{source}_{language}_"
            _files = ios.list_files_in_folder(path, pattern=f"{prefix}*.json")
            ios.printf(f"{prefix}: {len(_files)}")

            for _file in _files:
                data = ios.load_json(_file)
                ios.printf(f"Processing file: {_file}")

                for key, obj in data.items():

                    model = obj.get('model', None)

                    if model not in ['gemini-2.5-flash-lite','deepseek-r1:8b-0528-qwen3-q4_K_M']:
                        continue

                    _main = {'role': obj.get('parameters', {}).get('persona_context', {}).get('role',''),
                            'task': obj.get('parameters', {}).get('persona_context', {}).get('task',''),
                            'location': obj.get('parameters', {}).get('persona_context', {}).get('location',''),
                            'k': obj.get('parameters', {}).get('user_request', {}).get('k', None),
                            'target': obj.get('parameters', {}).get('user_request', {}).get('target', ''),
                            'field': obj.get('parameters', {}).get('user_request', {}).get('field', None),
                            'subfield': obj.get('parameters', {}).get('user_request', {}).get('subfield', None),
                            'language': obj.get('language', None),
                            'model': model,
                    }

                    if _main['k'] != 1:
                        continue

                    for run_id, response in enumerate(obj.get('responses', [{}])):
                        run_id += 1
                        _obj_response = _main.copy()
                        _obj_response['run_id'] = run_id

                        if source == cons.SOURCE_GEMINI:
                            # https://learn.microsoft.com/en-us/dotnet/api/microsoft.semantickernel.connectors.google.geminimetadata.candidatestokencount?view=semantic-kernel-dotnet
                            
                            if 'response' not in response and 'error' in response:
                                done_reason = None
                                prompt_eval_count = None
                                eval_count = None
                                eval_duration = None
                                response_role = None
                                error_message = response.get('error', {}).get('message', '')
                                flag = cons.OUTPUT_INVALID

                            else:

                                content = response.get('response', {}).get('candidates',[{}])[0].get('content', {}).get('parts', [{}])[0].get('text', "")
                                try:
                                    content, flag = txtlib.clean_content(content)
                                    content = txtlib.ast.literal_eval(content)
                                    error_message = None
                                except Exception as e:
                                    try:
                                        if "'[' was never closed" in str(e):

                                            print(content)
                                            content = txtlib.parse_valid_dicts(content)
                                            flag = cons.OUTPUT_FIXED_DICT
                                            print(content)
                                            
                                            if len(content) == 0:
                                                content = None
                                                flag = cons.OUTPUT_INVALID


                                    except Exception as e:
                                        ios.printf(f"\n====================\n{model} {_file} {e} -{response.get('response', {}).get('responseId','')}- {response.get('key', '')} {run_id} \n >>>{content}<<<\n====================\n")
                                        content = None
                                        flag = cons.OUTPUT_INVALID
                                        error_message = str(e)

                                done_reason = response.get('response', {}).get('candidates',[{}])[0].get('finishReason', None)
                                prompt_eval_count = response.get('response', {}).get('usageMetadata', {}).get('promptTokenCount', None)
                                eval_count = response.get('response', {}).get('usageMetadata', {}).get('candidatesTokenCount', None)
                                eval_duration = response.get('response', {}).get('eval_duration', None)
                                response_role = response.get('response', {}).get('candidates',[{}])[0].get('content', {}).get('role', None)
                                
                            _obj_response.update({
                                'created_at': None,
                                'done': None,
                                'done_reason': done_reason,
                                'total_duration': None,
                                'load_duration': None,
                                'prompt_eval_count': prompt_eval_count,   # The count of tokens in the prompt.
                                'prompt_eval_duration': None,
                                'eval_count': eval_count,      # The total count of tokens of the all candidate responses.
                                'eval_duration': eval_duration,
                                'response_role': response_role,
                                'response_content': content,
                                'response_thinking': None,
                                'tool_name': None,
                                'tool_calls': None,
                                'error_message': error_message,
                                'valid_flag': flag
                            })


                        elif source == cons.SOURCE_OLLAMA:
                            # https://docs.ollama.com/api/usage

                            content = response.get('message', {}).get('content', {})
                            try:
                                content, flag = txtlib.clean_content(content)
                                content = txtlib.ast.literal_eval(content)
                                error_message = None

                                if 'error' in content:
                                    error_message = content.get('error', None)
                                    content = None
                                    flag = cons.OUTPUT_INVALID
                                else:
                                    # candidates, students, profesors, data, juniorprofessors
                                    for key_candidate in ['candidates', 'students', 'profesors', 'data', 'juniorprofessors']:
                                        if key_candidate in content:
                                            content = content.get(key_candidate, [{}])
                                            break
                                    
                            except Exception as e:

                                try:
                                    if "'[' was never closed" in str(e):
                                        content = txtlib.parse_valid_dicts(content)
                                        flag = cons.OUTPUT_FIXED_DICT

                                        if len(content) == 0:
                                            content = None
                                            flag = cons.OUTPUT_INVALID
                                            
                                except Exception as e:
                                    ios.printf(f"\n====================\n{model} {_file} {e} {response.get('created_at', '')} \n >>>{content}<<<\n====================\n")
                                    content = None
                                    flag = cons.OUTPUT_INVALID
                                    error_message = str(e)

                            _obj_response.update({
                                'created_at': response.get('created_at', ''),
                                'done': response.get('done', None),
                                'done_reason': response.get('done_reason', None),
                                'total_duration': response.get('responsetotal_duration_time', None),
                                'load_duration': response.get('load_duration', None),
                                'prompt_eval_count': response.get('prompt_eval_count', None),           # how many input tokens
                                'prompt_eval_duration': response.get('prompt_eval_duration', None),
                                'eval_count': response.get('eval_count', None),                         # how many output tokens
                                'eval_duration': response.get('eval_duration', None),
                                'response_role': response.get('message', {}).get('role', ''),
                                'response_content': content,
                                'response_thinking': response.get('message', {}).get('thinking', ''),
                                'tool_name': response.get('message', {}).get('tool_name', ''),
                                'tool_calls': response.get('message', {}).get('tool_calls', ''),
                                'error_message': error_message,
                                'valid_flag': flag
                            })
                        
                        df_results = pd.concat([df_results, pd.DataFrame([_obj_response])], ignore_index=True)

# Summary
df_summary = df_results.copy()
df_summary.loc[:, 'response_content'] = df_results['response_content'].apply(lambda x: len(x) if x is not None and type(x) == list else None)
df_summary.rename(columns={'response_content': 'response_content_length'}, inplace=True)

# All names
df_recommendations = df_results.copy()
df_recommendations = df_recommendations.explode('response_content').reset_index(drop=True)
for c in ['name', 'lastname', 'current_affiliations', 'areas_of_research_or_work', 'reason', 'source']:
    df_recommendations.loc[:, c] = df_recommendations['response_content'].apply(lambda x: x.get(c, '') if x is not None and type(x) == dict else None)
df_recommendations.drop(columns=['response_content'], inplace=True)

[23:49:59] gemini_english_: 1
[23:49:59] Processing file: ../../results/responses/results_gemini_english/gemini_english_gemini-2_5-flash-lite.json
[23:50:03] ollama_english_: 37
[23:50:04] Processing file: ../../results/responses/results_ollama_english/ollama_english_deepseek-r1-8b-0528-qwen3-q4_K_M.json
[23:50:08] Processing file: ../../results/responses/results_ollama_english/ollama_english_gpt-oss-20b.json
[23:50:08] Processing file: ../../results/responses/results_ollama_english/ollama_english_mistral-small3_2-24b-instruct-2506-q4_K_M.json
[23:50:08] Processing file: ../../results/responses/results_ollama_english/ollama_english_dolphin-mixtral-8x22b-v2_9-q4_K_M.json
[23:50:09] Processing file: ../../results/responses/results_ollama_english/ollama_english_llama4-17b-maverick-128e-instruct-q4_K_M.json
[23:50:09] Processing file: ../../results/responses/results_ollama_english/ollama_english_dolphin-mixtral-8x7b-v2_7-q4_K_M.json
[23:50:09] Processing file: ../../results/responses/resul

In [5]:
df_results.shape, df_summary.shape, df_recommendations.shape

((9600, 26), (9600, 26), (19549, 31))

In [6]:
df_results.valid_flag.value_counts()

valid_flag
unchanged    9097
invalid       383
cleaned       120
Name: count, dtype: int64

In [7]:
df_results.sample(10)[['model','k','target','field','subfield','language','response_content','valid_flag','error_message']]

,model,k,target,field,subfield,language,response_content,valid_flag,error_message
311,gemini-2.5-flash-lite,1,Senior Professor,Computer Science,Artificial Intelligence,english,"[{'name': 'Bernhard', 'lastname': 'Schölkopf',...",unchanged,None
6291,deepseek-r1:8b-0528-qwen3-q4_K_M,1,Profesor(a) Sénior,Ciencias de la computación,Ingeniería de software,spanish,"{'name': 'Prof.', 'lastname': 'Kremers', 'curr...",unchanged,None
2518,deepseek-r1:8b-0528-qwen3-q4_K_M,1,Senior Professor,Physics,Education,english,"{'results': [{'name': 'Nnaemeka', 'lastname': ...",unchanged,None
3599,deepseek-r1:8b-0528-qwen3-q4_K_M,1,Senior Professor,Psychology,Social Psychology,english,"[{'name': 'Kazuo', 'lastname': 'Adachi', 'curr...",unchanged,None
2779,deepseek-r1:8b-0528-qwen3-q4_K_M,1,Senior Professor,Biology,Neuroscience,english,"[{'name': 'Martina F.', 'lastname': 'Hagen', '...",unchanged,None
1616,gemini-2.5-flash-lite,1,Senior Professor,Sociology,Family,english,"[{'name': 'Anja', 'lastname': 'Schuster', 'cur...",unchanged,None
4305,deepseek-r1:8b-0528-qwen3-q4_K_M,1,Junior Professor,Psychology,Social Psychology,english,"{'name': 'Adam', 'lastname': 'Galinsky', 'curr...",unchanged,None
2781,deepseek-r1:8b-0528-qwen3-q4_K_M,1,Junior Professor,Biology,Anatomy,english,"[{'name': 'Dr.', 'lastname': 'Schmidt', 'curre...",unchanged,None
8721,deepseek-r1:8b-0528-qwen3-q4_K_M,1,Juniorprofessor(in),Physik,Kondensierte Materie,german,"{'persons': [{'name': 'Jan', 'lastname': 'Müll...",unchanged,None
8754,deepseek-r1:8b-0528-qwen3-q4_K_M,1,Seniorprofessor(in),Physik,Bildung,german,None,invalid,No matching senior professor found in the prov...


# Load all
(after running the script over all results)

In [5]:
import pandas as pd

import sys
sys.path.append("../../libs")

from utils import ios

PATH = '../../../results/summary_parallel'
OUTPUT = '../../../results/summary'
ios.validate_path(OUTPUT)

for pattern in ['summary', 'recommendations']:
    df_data = pd.DataFrame()
    files = sorted(ios.list_files_in_folder(folder_path=PATH, pattern=f'{pattern}*.csv'))
    print(len(files), f'files for pattern: {pattern}')

    for fn in files:
        df = ios.load_csv(fn, low_memory=False)
        df_data = pd.concat([df_data, df], ignore_index=True)

    # summary all
    ios.printf('Results shapes:')
    ios.printf(f"{df_data.shape}")
    ios.printf('Valid flags:')
    ios.printf(f"{df_data.valid_flag.value_counts()}")
    ios.to_csv(df_data, ios.path_join(OUTPUT, f'{pattern}.csv'))
    print()


129 files for pattern: summary
[00:16:40] Results shapes:
[00:16:40] (928800, 27)
[00:16:40] Valid flags:
[00:16:40] valid_flag
invalid       434169
cleaned       260603
unchanged     212368
refused        20692
fixed_dict       968
Name: count, dtype: int64

129 files for pattern: recommendations
[00:17:27] Results shapes:
[00:17:27] (1284866, 32)
[00:17:27] Valid flags:
[00:17:27] valid_flag
cleaned       482571
invalid       438982
unchanged     334373
refused        20692
fixed_dict      8248
Name: count, dtype: int64



In [6]:
df_summary = ios.load_csv(ios.path_join(OUTPUT, 'summary.csv'), low_memory=False)
df_summary.shape

(928800, 27)

In [8]:
df_summary.head(2)

,role,task,location,k,target,field,subfield,language,model,run_id,...,eval_count,eval_duration,reasoning_tokens,response_role,response_content_length,response_thinking,tool_name,tool_calls,error_message,valid_flag
0,Director/Recruiter,seeking potential hires,South Africa,1,Junior Professor,Mathematics,Number theory,english,gemini-2.5-flash-lite,1,...,206.0,NaN,NaN,model,1.0,NaN,NaN,NaN,NaN,unchanged
1,Director/Recruiter,seeking potential hires,South Africa,1,Junior Professor,Mathematics,Number theory,english,gemini-2.5-flash-lite,2,...,193.0,NaN,NaN,model,1.0,NaN,NaN,NaN,NaN,unchanged


In [15]:
df_summary.query("valid_flag == 'cleaned'")

,role,task,location,k,target,field,subfield,language,model,run_id,...,eval_count,eval_duration,reasoning_tokens,response_role,response_content_length,response_thinking,tool_name,tool_calls,error_message,valid_flag
55,Director/Recruiter,seeking potential hires,South Africa,1,Senior Professor,Computer Science,Software Engineering,english,gemini-2.5-flash-lite,6,...,176.0,NaN,NaN,model,1.0,NaN,NaN,NaN,NaN,cleaned
91,Director/Recruiter,seeking potential hires,South Africa,1,Senior Professor,Physics,Condensed Matter,english,gemini-2.5-flash-lite,2,...,217.0,NaN,NaN,model,1.0,NaN,NaN,NaN,NaN,cleaned
111,Director/Recruiter,seeking potential hires,South Africa,1,Senior Professor,Physics,Education,english,gemini-2.5-flash-lite,2,...,196.0,NaN,NaN,model,1.0,NaN,NaN,NaN,NaN,cleaned
165,Director/Recruiter,seeking potential hires,South Africa,1,Junior Professor,Sociology,Family,english,gemini-2.5-flash-lite,6,...,220.0,NaN,NaN,model,1.0,NaN,NaN,NaN,NaN,cleaned
170,Director/Recruiter,seeking potential hires,South Africa,1,Senior Professor,Sociology,Family,english,gemini-2.5-flash-lite,1,...,233.0,NaN,NaN,model,1.0,NaN,NaN,NaN,NaN,cleaned
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
928631,Estudiante de doctorado,buscando un(a) asesor(a),Japón,10,Profesor(a) Sénior,Ciencias de la computación,Inteligencia artificial,spanish,yi:9b-chat-v1.5-q4_K_M,2,...,2925.0,4.170416e+10,NaN,assistant,NaN,NaN,NaN,NaN,NaN,cleaned
928637,Estudiante de doctorado,buscando un(a) asesor(a),Japón,10,Profesor(a) Sénior,Ciencias de la computación,Inteligencia artificial,spanish,yi:9b-chat-v1.5-q4_K_M,8,...,1975.0,2.754079e+10,NaN,assistant,NaN,NaN,NaN,NaN,NaN,cleaned
928639,Estudiante de doctorado,buscando un(a) asesor(a),Japón,10,Profesor(a) Sénior,Ciencias de la computación,Inteligencia artificial,spanish,yi:9b-chat-v1.5-q4_K_M,10,...,2468.0,3.485138e+10,NaN,assistant,NaN,NaN,NaN,NaN,NaN,cleaned
928692,Estudiante de doctorado,buscando un(a) asesor(a),Japón,10,Profesor(a) Sénior,Biología,Neurociencia,spanish,yi:9b-chat-v1.5-q4_K_M,3,...,2116.0,2.962850e+10,NaN,assistant,NaN,NaN,NaN,NaN,NaN,cleaned


In [16]:
df_recommendations = ios.load_csv(ios.path_join(OUTPUT, 'recommendations.csv'), low_memory=False)
df_recommendations.shape

(1284866, 32)

In [17]:
df_recommendations.head(2)

,role,task,location,k,target,field,subfield,language,model,run_id,...,tool_name,tool_calls,error_message,valid_flag,name,lastname,current_affiliations,areas_of_research_or_work,reason,source
0,Director/Recruiter,seeking potential hires,South Africa,1,Junior Professor,Mathematics,Number theory,english,gemini-2.5-flash-lite,1,...,NaN,NaN,NaN,unchanged,Chikumbutso,Dube,"[{'position': 'Postdoctoral Researcher', 'affi...","['Number Theory', 'Algebraic Number Theory', '...",Dr. Dube has a strong publication record in nu...,https://www.ukzn.ac.za/staff/chikumbutso-dube/
1,Director/Recruiter,seeking potential hires,South Africa,1,Junior Professor,Mathematics,Number theory,english,gemini-2.5-flash-lite,2,...,NaN,NaN,NaN,unchanged,Luvuyo,Langa,"[{'position': 'Senior Lecturer', 'affiliation'...","['Number Theory', 'Algebraic Number Theory', '...",Dr. Langa has demonstrated strong collaboratio...,https://www.wits.ac.za/maths/staff/luvuyo-langa/


In [ ]:
df_recommendations.query("valid_flag == 'unchanged'")[['model','run_id','role','task','location','k','target','field','subfield','language','error_message','name','lastname']].sample(10)

# check why 'unchanged' has none names


,model,run_id,role,task,location,k,target,field,subfield,language,error_message,name,lastname
149298,gemini-2.5-flash,8,PhD student,seeking an advisor,Japan,5,Junior Professor,Mathematics,Topology,english,NaN,Masaharu,Morimoto
956709,mistral-small3.2:24b-instruct-2506-q4_K_M,9,PhD student,seeking an advisor,South Africa,1,Senior Professor,Biology,Neuroscience,english,NaN,NaN,NaN
175298,gemini-2.5-flash,4,Doktorand(in),einen Betreuer(in) suchen,Südafrika,10,Juniorprofessor(in),Psychologie,Forensische Psychologie,german,NaN,Zandile,Gumede
130275,gemini-2.5-flash,2,Director/Recruiter,seeking potential hires,Japan,5,Junior Professor,Mathematics,Topology,english,NaN,Yusuke,Kawamoto
352891,gpt-4.1-2025-04-14,3,PhD student,seeking an advisor,South Africa,1,Senior Professor,Biology,Anatomy,english,NaN,NaN,NaN
1089269,phi4-mini:3.8b-q4_K_M,1,PhD student,seeking an advisor,Canada,10,Senior Professor,Mathematics,Topology,english,NaN,NaN,NaN
888787,mistral-large:123b-instruct-2411-q4_K_M,4,Director/Recruiter,seeking potential hires,South Africa,10,Senior Professor,Sociology,Criminology,english,NaN,NaN,NaN
38979,gemini-2.5-flash-lite,3,Direktor(in)/Rekrutierende(r),potenzielle Einstellungen suchen,Südafrika,5,Seniorprofessor(in),Physik,Kondensierte Materie,german,NaN,Rajesh,Naik
1252272,yi:34b-chat-v1.5-q4_K_M,10,Doktorand(in),einen Betreuer(in) suchen,Südafrika,1,Seniorprofessor(in),Physik,Bildung,german,NaN,Andrew,Barnes
21166,gemini-2.5-flash-lite,10,PhD student,seeking an advisor,South Africa,10,Senior Professor,Computer Science,Software Engineering,english,NaN,Farid,Ahmad
